In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from src.data import PoseDataset, prepare_pose_graph_batch

ann_path = "data/crowdpose/annotations/crowdpose_val.json"
img_root = "data/crowdpose/images"
dataset = PoseDataset.from_path(ann_path, img_root)
dataloader = dataset.create_dataloader(batch_size=4, shuffle=False)

ex_batch = next(iter(dataloader))
prepare_pose_graph_batch(ex_batch)
print("Batch keys:", list(ex_batch.keys()))
print("node_pos shape:", ex_batch["node_pos"].shape)   # (B*N*K, 2)
print("edge_index shape:", ex_batch["edge_index"].shape)


Batch keys: ['image_id', 'img_path', 'keypoints', 'visibility', 'edge_index', 'node_pos', 'batch', 'batch_idx', 'pose_idx', 'kpt_type_idx', 'node_idx', 'canonical_visibility', 'root_pos', 'scale']
node_pos shape: torch.Size([60, 2])
edge_index shape: torch.Size([2, 136])


In [3]:
# --- Diffusion model: forward (add noise) + backward (denoise) ---
from src.model import KeypointDiffusionModel, KeypointDiffusionConfig

device = "cuda" if torch.cuda.is_available() else "cpu"

config = KeypointDiffusionConfig(dim_hidden=128, num_layers=4)
model = KeypointDiffusionModel(config, num_keypoints=15).to(device)

node_pos     = ex_batch["node_pos"].to(device)          # (B*N*K, 2)
edge_index   = ex_batch["edge_index"].to(device)        # (2, B*N*E)
kpt_type_idx = ex_batch["kpt_type_idx"].reshape(-1).to(device)   # (B*N*K,)
batch_idx    = ex_batch["batch"].to(device)             # (B*N*K,)
B            = ex_batch["keypoints"].shape[0]

# Forward pass: add noise at a random timestep
t = torch.randint(1, config.num_train_steps + 1, (B,), device=device)
noisy_pos, noise = model.add_noise(node_pos, t, batch_idx)
print("noisy_pos:", noisy_pos.shape)

# Backward pass: predict noise
pred_noise = model.backward_process(
    noisy_node_pos=noisy_pos,
    edge_index=edge_index,
    kpt_type_idx=kpt_type_idx,
    batch_idx=batch_idx,
    timestep=t,
)
loss = torch.nn.functional.mse_loss(pred_noise, noise)
print(f"Noise prediction loss (random init): {loss.item():.4f}")

# DDIM sampling from pure noise
N = ex_batch["keypoints"].shape[1]
K = 15
num_nodes = B * N * K
sampled = model.ddim_sample(
    edge_index=edge_index,
    kpt_type_idx=kpt_type_idx,
    batch_idx=batch_idx,
    num_nodes=num_nodes,
    num_inference_steps=20,
    device=device,
)
print("DDIM sample shape:", sampled.shape)  # (B*N*K, 2)


noisy_pos: torch.Size([60, 2])
Noise prediction loss (random init): 1.0670
DDIM sample shape: torch.Size([60, 2])


In [4]:
# --- DINOv3: embed the first image in the batch ---
import torchvision
from transformers import AutoImageProcessor, AutoModel

pretrained_model_name = "facebook/dinov3-vits16-pretrain-lvd1689m"
processor = AutoImageProcessor.from_pretrained(pretrained_model_name, use_fast=True)
dino = AutoModel.from_pretrained(pretrained_model_name).to(device)

# Load the first image from the batch
ex_path = ex_batch["img_path"][0]
ex_img = torchvision.io.image.read_image(ex_path)   # (3, H, W) uint8
print("Image shape:", ex_img.shape)

# Convert to PIL for the processor (it expects PIL or numpy)
from torchvision.transforms.functional import to_pil_image
pil_img = to_pil_image(ex_img)

inputs = processor(images=pil_img, return_tensors="pt").to(device)
with torch.inference_mode():
    outputs = dino(**inputs)

# pooler_output: (1, hidden_dim) — global CLS token embedding
pooled = outputs.pooler_output
print("DINOv3 pooled embedding shape:", pooled.shape)

# last_hidden_state: (1, num_patches+1, hidden_dim) — patch-level features
patch_feats = outputs.last_hidden_state[:, 1:]   # drop CLS token
print("DINOv3 patch features shape:", patch_feats.shape)


config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/86.4M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Image shape: torch.Size([3, 835, 550])
DINOv3 pooled embedding shape: torch.Size([1, 384])
DINOv3 patch features shape: torch.Size([1, 200, 384])
